# Topic Modeling

In [ ]:
# Documentation source : https://axlmrin.github.io/bertopic-tutorial/tutorial.html#some-good-practices

# Sur les librairies à installer, leur comptatibilité et l'environnement cf : environnement_république

# Sur le terminal de VS-code : 
# Commencer par activer l'environnement virtuel conda : conda activate republique_cest_quoi
# python -m spacy download fr_core_news_sm
# python -m spacy download fr_dep_news_trf

In [ ]:
# Commencer par bien sélectionner le bon kernel "république_c'est_quoi"

In [ ]:
# Importer l'intégralité des librairies potentiellement nécessaires 

import pandas as pd
import numpy as np
import spacy
from bertopic import BERTopic
import umap
from hdbscan import HDBSCAN

from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer


In [ ]:
df = pd.read_csv(
    "../data/interim/df_repu_regroup.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [ ]:
import datetime
import locale

# Active la locale française (nécessaire pour le format)
locale.setlocale(locale.LC_TIME, "fr_FR.UTF-8")

In [ ]:
df["dateSeance_ts"] = pd.to_datetime(df["dateSeanceJour"], format="%A %d %B %Y")
df["dateSeance_day"] = df["dateSeance_ts"].dt.normalize()  

In [ ]:
docs = df["Texte_clean"].tolist()
language = "french"
language_short = language[:2]

***Possible de faire Étapes 1 et 2 sur AT et ensuite fusionner pour faire analyse***

## Étape 1 :  Paramètrages du Bertopic

### 1. Embedding 

#### Choisir le modèle d'embedding

--> aller voir plus tout ce qui a été fait dans le ppt et Axel Morin pour repréciser les effets de chaque, et ce que ça implique

In [ ]:
embedding_model = SentenceTransformer(
    "Lajavaness/sentence-camembert-large", trust_remote_code=True # top 1 sur données AN
    # "Lajavaness/sentence-flaubert-base", top 2 sur données AN
    #"all-MiniLM-L6-v2",  # fonctionne ok sur petits df, très rapide
)

print("device used :", embedding_model.device)  # Vérifie si le modèle est sur GPU ou CPU
# Comment intérprêter résultat ==> mps:0 = GPU Apple Silicon (Metal)

In [ ]:
# Modèles à charger en interne car pas directement utilisables 
    # "flaubert-large-cased", # https://huggingface.co/flaubert/flaubert_large_cased
    # "camembert/camembert-large", # https://huggingface.co/almanach/camembert-large
    # "jinaai/jina-embeddings-v3", # 8192 tokens with RoPE. https://huggingface.co/jinaai/jina-embeddings-v3

####  Entrainer et sauvegarder les embeddings (par corpus et modèle d'embedding)

In [ ]:
# 1 Calculer les embeddings (valable uniquement pour un corpus)
# embeddings = embedding_model.encode(
    #docs,
    #show_progress_bar=True,
    #batch_size=32
#)

In [ ]:
# 2. Sauvegarder ces embeddings 
#np.save("./data/embeddings_df_regroup.npy", embeddings)

In [ ]:
# 3. Recharger les embeddings 
embeddings = np.load("./data/embeddings_df_regroup.npy")

### 2/3. Choisir la granularité du modèle

Once you've chosen an embedding model, you can change the `n_neighbors` and `min_cluster_size`. Both work jointly: ***the lower these paramters, the smaller grain and more specific the topics***. 

To change these parameters, one must explicitly declare `UMAP` and `HDBSCAN` objects and pass them on to the `BERTopic` model:

In [ ]:
# Paramètres du modèles
hdbscan_model = HDBSCAN(
    min_cluster_size=10, # The minimum number of elements in a group to be considered a cluster, otherwise, it’s considered as noise. Start with a rather large value and reduce incrementally until obtaining a fine grained topic model
    metric="euclidean", # This parameter defines the metric used to quantify the distance between 2 docs
    cluster_selection_method="eom",
    prediction_data=True # Nécessaire pour stats intégrales car permet de calculer les probabilités de chaque thème par discours (= pas binaire)
)
umap_model = umap.UMAP(
    n_neighbors=10, # This parameter defines what’s considered as “close”. Start with a rather large value and reduce incrementally until obtaining the desired level of topics specificity
    metric="cosine", # This parameter defines the metric used to quantify the similarity between 2 vectors
    n_components=2, # Defines the number of dimensions of the output space. Start with n_components=5. Increase when the topic model does not grasp semantic subtelties. Decrease when the topic model focuses on non-essential disparities of your corpus.
    min_dist=0.0, # This parameter is “essentially aesthetic”, low values will create denser clusters
    random_state=42, # Permet la reproductibilité (42 car chiffre souvent utilisé, sinon tester random_state=RANDOM_SEED)
    low_memory=False
)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True) # reduce the impact of frequent word so with it we retrieve words that make a group unique!


### 4. Tokenizer

### 5. Mise en forme des mots (stopwords)

In [ ]:
### Option 1 : cf-tfidf

In [ ]:
### Option 2 Avec spacy ### 
nlp = spacy.load("fr_core_news_sm")  # !python -m spacy download fr_core_news_sm
french_stopwords = list(nlp.Defaults.stop_words)
vectorizer_model = CountVectorizer(stop_words=french_stopwords)

### 7. Entrainer le modèle / fine-tune

In [ ]:
# Créer le modèle
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,  # remove stopwords after embbedings
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True,
    ctfidf_model=ctfidf_model 
)

# emb = model.encode(list_of_sentences, batch_size=64, show_progress_bar=True)

In [ ]:
# Fiter le modèle
topics, probs = topic_model.fit_transform(documents=docs, embeddings=embeddings)  # rajouter embeddings pour éviter de les recalculer (phase la plus longue et qui ne change pas si on ne bouge pas de corpus et modèle d'embedding)

## Étape 2 : Raffiner le modèle

### Premières visualisations

In [ ]:
# Recharger si besoin directement 
embeddings = np.load("./data/embeddings_df_regroup.npy") #potentiellement nécessaire plus tard pour des analyses plus poussées
topic_model = BERTopic.load("./data/10-10-2-with-ctfidf", embedding_model=embedding_model)

In [ ]:
table_topic = topic_model.get_topic_info()
table_topic[:180]

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(docs)

In [ ]:
# Visualisation hiérarchique (dendogramme)

fig_hierarchical = topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)
fig_hierarchical

In [ ]:
# Visualiser la répartition des documents dans l'espace 
(topic_model
    .visualize_documents(
        docs = docs,
        embeddings = embeddings,
        hide_annotations = True, # better readability
        topics = [96, 76, 104, 169, 50, 8]        # Select topics to highlight
        # height = 300, # Adjust the height of the plot
        # width = 800 # Adjust the width of the plot 
    )
)

In [ ]:
# Visualiser les mots par topics 
topic_model.visualize_barchart(
    n_words = 20, # Select the number of words to display per topic
    topics = [12] , # Select specific topics to display
    # top_n_topics = 10, # Select the first n topics to display
    # height = 300, # Adjust the height of the plot
    # width = 800 # Adjust the width of the plot 
)

In [ ]:
# Optional: visualize
fig_topic_distance_map = topic_model.visualize_topics()
fig_topic_distance_map

In [ ]:
# ~ 6MB
topic_model.save(
    path = "./data/15-15-7-with-ctfidf",
    serialization = "safetensors",
    save_ctfidf = True
)

### Merger les topics

- 2 options : manuellement ou automatiquement 

--> Le merge automatique plante fréquemment. 

--> Ici volonté d'un merge manuel (supervisé). On peut alors opérer étapes par étapes, ou tout d'un coup. Il vaut mieux tout faire d'un coup pour une reproductibilité plus claire, puisqu'à chaque fois cela recalcule les proximités et change les visualisations.

In [ ]:
# V1 extensive (22 topics cohérents, le + solide même si peut-être trop large)
topics_to_merge = [96, 76, 104, 169, 50, 8], [110, 93, 139, 46, 144, 128, 72, 103, 12] , [121, 66, 85, 75], [146, 117, 97, 115, 87, 41, 130, 161], [158, 71,  95, 4, 65, 92, 151, 118, 120], [148, 17, 88, 83, 28, 1, 7, 53, 78, 29, 64], [5, 150, 73, 67, 174, 116], [9, 98], [108, 111, 56, 94, 82, 131, 134], [173, 153, 159, 70, 58, 25, 33, 126, 90, 142, 34, 137, 167, 81, 74, 133], [113, 42, 86, 176, 149], [135, 122, 132, 143, 45, 61, 47, 13, 44, 127, 125, 14, 49, 101, 175, 136, 80, 102, 147], [35, 39, 62, 3, 168, 164, 140, 165, 172, 141, 129, 37, 105, 177, 112], [152, 21, 11, 107, 156, 22], [68, 31, 26, 84], [69, 36, 48, 77, 24, 57, 79], [32, 16, 18], [6, 19, 2, 23, 20, 100, 160], [89, 99, 114, 15, 145, 162,  54, 52, 38, 60, 124, 119, 157, 171, 138], [166, 55, 51, 30, 59, 43, 27, 40, 63, 91, 154, 106, 155, 109, 178], [163, 10, 123, 170] 
topic_model.merge_topics(docs, topics_to_merge)

In [ ]:
# V2 plus restreinte (attention, à partir des topics originels)
topics_to_merge = [135, 122, 132, 143, 45, 61, 47, 13, 44, 127, 125, 14, 49, 69, 36, 48, 77, 24, 57, 79, 163, 10, 123, 170, 101, 175, 136, 80, 102, 147], []

In [ ]:
table_topic = topic_model.get_topic_info()
table_topic[:40]

In [ ]:
fig_hierarchical = topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)
fig_hierarchical

In [ ]:
# # Les principales fonctions à tester pour avoir un aperçu simple :

# topic_model.get_topic_info()
# topic_model.visualize_barchart()
# topic_model.visualize_topics()
# topic_model.visualize_hierarchy()
# topic_model.visualize_documents(df["texte"].to_list())

In [ ]:
topic_model.visualize_barchart(
    n_words = 10, # Select the number of words to display per topic
    topics = [21], # Select specific topics to display
    # top_n_topics = 6, # Select the first n topics to display
    # height = 300, # Adjust the height of the plot
    # width = 800 # Adjust the width of the plot 
)

In [ ]:
# Optional: visualize
fig_topic_distance_map = topic_model.visualize_topics()
fig_topic_distance_map

### Possible de Réattribuer le "bruit" à d'autres topics --> fails 

In [ ]:
topics_reduced = topic_model.reduce_outliers(
    documents = docs, 
    topics = topics, 
    probabilities= probs, 
    embeddings = embeddings,
    strategy="embeddings" # ou probabilites ou cf-tf-idf
)

In [ ]:
print(type(topics_reduced))  # Doit afficher <class 'list'> ou <class 'numpy.ndarray'>
print(topics_reduced[:10])   # Affiche les 10 premiers topics réduits

In [ ]:
topic_model.visualize_barchart(topics=topics_reduced, n_words=10)

In [ ]:
# Vérifiez les entrées de reduce_outliers
print(type(docs), len(docs))          # Doit être une liste ou un tableau de documents
print(type(topics), len(topics))      # Doit être une liste ou un tableau de topics
print(type(probs), probs.shape)       # Doit être un tableau de probabilités
print(type(embeddings), embeddings.shape)  # Doit être un tableau d'embeddings

# Si l'une de ces entrées est incorrecte, corrigez-la avant d'appeler reduce_outliers


In [ ]:
# Mettre à jour le modèle avec les topics réduits
topic_model.update_topics(docs, topics=topics_reduced, embeddings=embeddings)

# Récupérer les informations sur les topics réduits
reduced_topic_info = topic_model.get_topic_info()
print(reduced_topic_info.head())


In [ ]:
topic_model.visualize_heatmap(n_clusters=len(set(topics_reduced)))


In [ ]:
table_topic = topics_reduced.get_topic_info()
table_topic[:40]

In [ ]:
fig_hierarchical = topics_reduced.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)
fig_hierarchical

### Nommer les thématiques ainsi construites 

In [ ]:
table_topic = topic_model.get_topic_info()
table_topic[:40]

In [ ]:
topic_labels = {
    0: "Laïcité-Islam",
    1: "École",
    2: "Outre-mers",
    3: "Territoires métropolitains", 
    4: "Fiscalité-travail",
    5: "Maintien de l'ordre",
    6: "Modèle démocratique",
    7: "Motions de censure",
    8: "Immigration",
    9: "Associatif jeunesse",
    10: "Santé",
    11: "Politique étrangère",
    12: "Justice",
    13: "Champ 'républicain'",
    14: "Agri-éco",
    15: "Habitat",
    16: "Droits corporels",
    17: "Données numériques",
    18: "Terrorisme",
    19: "Particularismes linguistiques",
    20: "Antisémitisme",
    21: "Mémoire"
}


In [ ]:
topic_model.set_topic_labels(topic_labels)

### Exporter les résultats dans le df

In [ ]:
df["topic"] = topic_model.topic_mapper_.map_topics(topics)

In [ ]:
topic_model.topic_mapper_.topic_mapping_

#### Méthode bis : tout via pandas hors du topic modelling

In [ ]:
# Reprendre les merges de topic, en mettant bien toutes les catégories (y compris celles non-mergées) ATTENTION ICI NE REFAIT PAS DE CATÉGORIES ORDONNÉES DU PLUS HAUT AU PLUS BAS D'OÙ CHANGEMENT ORDRE PRÉSENTATION
merges = [[-1], [148, 17, 88, 83, 28, 1, 7, 53, 78, 29, 64], [135, 122, 132, 143, 45, 61, 47, 13, 44, 127, 125, 14, 49, 101, 175, 136, 80, 102, 147], [6, 19, 2, 23, 20, 100, 160, 138], [166, 55, 51, 30, 59, 43, 27, 40, 63, 91, 154, 106, 155, 109, 178], [35, 39, 62, 3, 168, 164, 140, 165, 172, 141, 129, 37, 105, 177, 112], [173, 153, 159, 70, 58, 25, 33, 126, 90, 142, 34, 137, 167, 81, 74, 133], [89, 99, 114, 15, 145, 162,  54, 52, 38, 60, 124, 119, 157, 171], [0], [158, 71,  95, 4, 65, 92, 151, 118, 120], [69, 36, 48, 77, 24, 57, 79], [152, 21, 11, 107, 156, 22], [110, 93, 139, 46, 144, 128, 72, 103, 12], [5, 150, 73, 67, 174, 116], [96, 76, 104, 169, 50, 8], [32, 16, 18], [68, 31, 26, 84], [146, 117, 97, 115, 87, 41, 130, 161], [108, 111, 56, 94, 82, 131, 134], [9, 98], [163, 10, 123, 170], [113, 42, 86, 176, 149], [121, 66, 85, 75]]

In [ ]:
topic_labels_DF = {
    0: "Bruit",
    1: "Laïcité-Islam",
    2: "École",
    3: "Outre-mers",
    4: "Territoires métropolitains", 
    5: "Fiscalité-travail",
    6: "Maintien de l'ordre",
    7: "Modèle démocratique",
    8: "Motions de censure",
    9: "Immigration",
    10: "Associatif jeunesse",
    11: "Santé",
    12: "Politique étrangère",
    13: "Justice",
    14: "Champ 'républicain'",
    15: "Agri-éco",
    16: "Habitat",
    17: "Droits corporels",
    18: "Données numériques",
    19: "Terrorisme",
    20: "Particularismes linguistiques",
    21: "Antisémitisme",
    22: "Mémoire",
}


In [ ]:
topic_mapping = {}

for new_id, group in enumerate(merges):
    for old_id in group:
        topic_mapping[old_id] = new_id


In [ ]:
new_topics = [topic_mapping[t] for t in topics]
df["topic"] = new_topics

In [ ]:
# Générer les colonnes booléennes
dummies = pd.get_dummies(df["topic"], prefix="topic")

# Renommer les colonnes pour plus de clarté (optionnel)
dummies.columns = [f"topic_{i}" for i in range(-1, 22)]
dummies = dummies.reindex(columns=[f"topic_{i}" for i in range(-1, 22)], fill_value=False)

# S'assurer que toutes les colonnes de -1 à 22 sont présentes
for i in range(-1, 22):
    col_name = f"topic_{i}"
    if col_name not in dummies.columns:
        dummies[col_name] = False

# Réorganiser les colonnes dans l'ordre
dummies = dummies[[f"topic_{i}" for i in range(-1, 22)]]

# Concaténer avec le DataFrame original
df = pd.concat([df, dummies], axis=1)

# Afficher le résultat
print(df)

In [ ]:
# Afficher le résultat
print(dummies)

In [ ]:
# Ajouter une colonne avec les nouveaux noms
df["topic_label"] = df["topic"].replace(topic_labels_DF)

In [ ]:
nouveaux_noms = ["bruit", "Laïcité-Islam", "École", "Outre-mers", "Territoires métropolitains",  "Fiscalité-travail", "Maintien de l'ordre", "Modèle démocratique", "Motions de censure", "Immigration", "Associatif jeunesse", "Santé", "Politique étrangère", "Justice", "Champ 'républicain'", "Agri-éco", "Habitat", "Droits corporels", "Données numériques", "Terrorisme", "Particularismes linguistiques", "Antisémitisme", "Mémoire"]

In [ ]:
# Création du dictionnaire de renommage
anciens_noms = [f"topic_{i}" for i in range(-1, 23)]
dico_renommage = dict(zip(anciens_noms, nouveaux_noms))

# Renommage des colonnes
df = df.rename(columns=dico_renommage)


In [ ]:
df

In [ ]:
df.to_csv("../data/interim/topic_falses.csv", index=False)

In [ ]:
# Sauvegarder le modèle une fois les regroupements effectués 

# ~ 500 KB
topic_model.save(
    path = "./bertopic-default",
    serialization = "safetensors",
    save_ctfidf = False
)

# ~ 6MB
topic_model.save(
    path = "./bertopic-default-with-ctfidf",
    serialization = "safetensors",
    save_ctfidf = True
)

In [ ]:
topic_model.save("TM_Test_Regroup_Camembert_20")

## Étape 3 : Analyses exploratoires

In [ ]:
from bertopic import BERTopic
topic_model = BERTopic.load("../data/interim/TM_Test_Regroup_Camembert_20")


### Analyses diachroniques

* `global_tuning` : Tuning général
  * Indique s'il faut calculer la moyenne de la représentation d'un sujet à l'instant *t* avec sa représentation globale.
* `evolution_tuning` : Tuning évolutif
  * Indique s'il faut calculer la moyenne de la représentation d'un sujet à l'instant *t* avec la représentation de ce sujet à l'instant *t-1*.
* `nr_bins`
  * Nombre de compartiments dans lesquels placer les horodatages. Il est inefficace sur le plan informatique d'extraire les sujets à des milliers d'horodatages différents. Il est donc conseillé de maintenir cette valeur en dessous de 20.


In [ ]:
# Évolution générale 
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps=df["dateSeance_day"],
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=100,
)

In [ ]:
fig_dynamic_topic = topic_model.visualize_topics_over_time(
    topics_over_time, top_n_topics=3
)
fig_dynamic_topic

In [ ]:
# Analyse annuelle 
df["year"] = df["dateSeance_day"].dt.year
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps=df["year"],
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=8,  # Une bin par année si votre période couvre 10 ans
)


In [ ]:
fig_dynamic_topic = topic_model.visualize_topics_over_time(
    topics_over_time, top_n_topics=3
)
fig_dynamic_topic

In [ ]:
# Analyse mensuelle 
df["year_month"] = df["dateSeance_day"].dt.to_period('M')
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps=df["year_month"].astype(str),  # Convertir en chaîne pour éviter les erreurs
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=96,  # 8 ans * 12 mois
)


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig = topic_model.visualize_topics_over_time(topics_over_time)
plt.show()

In [ ]:
fig_dynamic_topic = topic_model.visualize_topics_over_time(
    topics_over_time, top_n_topics=23
)
fig_dynamic_topic

In [ ]:
# Analyse hebdomadaire
df["year_week"] = df["dateSeance_day"].dt.to_period('W')
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps=df["year_week"].astype(str),
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=416,  # 8*52
)


In [ ]:
# Supposons que df est votre dataframe et docs est une liste de textes
docs = df["Texte_clean"].tolist()  # Assurez-vous que docs est aligné avec df

# Définir la période
start_date = "2020-01-01"
end_date = "2022-12-31"
mask = (df["dateSeance_day"] >= start_date) & (df["dateSeance_day"] <= end_date)
filtered_df = df.loc[mask]

# Filtrer docs et timestamps
filtered_docs = filtered_df[docs].tolist()  # ou .values si nécessaire
filtered_timestamps = filtered_df["dateSeance_day"]

# Appeler topics_over_time
topics_over_time = topic_model.topics_over_time(
    filtered_docs,
    timestamps=filtered_timestamps,
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=50,
)


In [ ]:
start_date = "2021-02-01"
end_date = "2021-07-30"
mask = (df["dateSeance_day"] >= start_date) & (df["dateSeance_day"] <= end_date)
filtered_docs = [doc for doc, keep in zip(docs, mask) if keep]  # Filtrer docs
filtered_timestamps = df.loc[mask, "dateSeance_day"]  # Filtrer timestamps

topics_over_time = topic_model.topics_over_time(
    filtered_docs,
    timestamps=filtered_timestamps,
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=20,  # À ajuster selon la durée de la période
)


In [ ]:
fig_dynamic_topic = topic_model.visualize_topics_over_time(
    topics_over_time, top_n_topics=3
)
fig_dynamic_topic

### Topics par groupe parlementaire

In [ ]:
topics_per_class = topic_model.topics_per_class(
    docs, classes=df["groupe&gvt_affiliation"]
)

In [ ]:
fig_topics_per_class = topic_model.visualize_topics_per_class(
    topics_per_class, top_n_topics=15
)
fig_topics_per_class

### Topics par partis

In [ ]:
import pandas as pd

df_stat = topics_per_class.copy()
df_stat = df_stat[df_stat["Topic"] != -1]

# normalisation par parti (recommandée)
df_stat["prop"] = (
    df_stat["Frequency"]
    / df_stat.groupby("Class")["Frequency"].transform("sum")
)


In [ ]:
df_stat["Topic_name"] = df_stat["Topic"].map(topic_labels)


In [ ]:
df_stat

In [ ]:
TOP_N = 13

df_top = (
    df_stat.sort_values(["Class", "prop"], ascending=[True, False])
      .groupby("Class")
      .head(TOP_N)
)


In [ ]:
import plotly.graph_objects as go

parties = df_top["Class"].unique()

fig = go.Figure()

# Ajouter une trace par parti (une seule visible au départ)
for i, party in enumerate(parties):
    df_p = df_top[df_top["Class"] == party]

    fig.add_trace(
        go.Bar(
            x=df_p["prop"],
            y=df_p["Topic_name"],
            orientation="h",
            visible=(i == 0),
            name=party
        )
    )

# Boutons du menu déroulant
buttons = []
for i, party in enumerate(parties):
    visibility = [False] * len(parties)
    visibility[i] = True

    buttons.append(
        dict(
            label=party,
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Top topics – {party}"}
            ]
        )
    )

fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.15,
        y=1
    )],
    title=f"Top topics – {parties[0]}",
    xaxis_title="Importance relative",
    yaxis_title="Topics",
    height=500,
    margin=dict(l=200)
)

fig.show()


### Répartitions des partis par topic

## Étape 4 : Analyse df via stats classiques

## Étape 5 : Analyses via matrices des probabilités et pas topic unitaire  

In [ ]:
# Réimporter embeddings
embeddings = np.load("./data/embeddings_df_regroup.npy")

In [ ]:
# Paramètres du modèles
hdbscan_model = HDBSCAN(
    min_cluster_size=10, # The minimum number of elements in a group to be considered a cluster, otherwise, it’s considered as noise. Start with a rather large value and reduce incrementally until obtaining a fine grained topic model
    metric="euclidean", # This parameter defines the metric used to quantify the distance between 2 docs
    cluster_selection_method="eom",
    prediction_data=True # Nécessaire pour stats intégrales car permet de calculer les probabilités de chaque thème par discours (= pas binaire)
)
umap_model = umap.UMAP(
    n_neighbors=10, # This parameter defines what’s considered as “close”. Start with a rather large value and reduce incrementally until obtaining the desired level of topics specificity
    metric="cosine", # This parameter defines the metric used to quantify the similarity between 2 vectors
    n_components=2, # Defines the number of dimensions of the output space. Start with n_components=5. Increase when the topic model does not grasp semantic subtelties. Decrease when the topic model focuses on non-essential disparities of your corpus.
    min_dist=0.0, # This parameter is “essentially aesthetic”, low values will create denser clusters
    random_state=42, # Permet la reproductibilité (42 car chiffre souvent utilisé, sinon tester random_state=RANDOM_SEED)
    low_memory=False
)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True) # reduce the impact of frequent word so with it we retrieve words that make a group unique!

In [ ]:
# Recréer le modèle
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,  # remove stopwords after embbedings
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True,
    ctfidf_model=ctfidf_model 
)

# Faire tourner le modèle
topics, probs = topic_model.fit_transform(documents=docs, embeddings=embeddings)  # rajouter embeddings pour éviter de les recalculer (phase la plus longue et qui ne change pas si on ne bouge pas de corpus et modèle d'embedding)

In [ ]:
probs.shape
# (n_docs, n_topics)

Plus bas à corriger et revoir 

In [ ]:
df_doc = topic_model.get_document_info(docs)

In [ ]:
# Éviter le soucis de doublons
df_meta = (
    df.drop_duplicates(subset="id_syceron")
)

In [ ]:
# Fusionner
df_full = df_doc.merge(df_meta, on="doc_id", how="left")

In [ ]:
# Construire la matrice « député × topics » par probabilité

import numpy as np
import pandas as pd

topic_labels = [f"T{i}" for i in range(len(probs[0]))]

df_probs_députés = pd.DataFrame(probs, columns=topic_labels)
df_probs_députés["nom_orateur_clean"] = df_full["nom_orateur_clean"].values

df_topic_depute = df_probs_députés.groupby("nom_orateur_clean").mean()


In [ ]:
# Construire la matrice « groupe × topics » par probabilité
topic_labels = [f"T{i}" for i in range(len(probs[0]))]

df_probs_groupes = pd.DataFrame(probs, columns=topic_labels)
df_probs_groupes["groupe&gvt_affiliation"] = df_full["groupe&gvt_affiliation"].values

df_topic_groupe = df_probs_groupes.groupby("groupe&gvt_affiliation").mean()

In [ ]:
# Enlever le -1
df_topic_depute = df_topic_depute.drop(columns=[-1], errors="ignore")
df_topic_groupe = df_topic_groupe.drop(columns=[-1], errors="ignore")

# Vérifier la masse thématique 
df_topic_depute.sum(axis=1).describe()
df_topic_groupe.sum(axis=1).describe()

In [ ]:
# Visualisation rapide 

import seaborn as sns
import matplotlib.pyplot as plt

sns.clustermap(
    df_topic_groupe,
    metric="cosine",
    method="average",  # ou "complete"
    cmap="viridis"
)


In [ ]:
theme_names = {
    "topic_0": "territoires",
    "topic_1": "laicité",
    "topic_2": "éducation",
    "topic_3": "répressif",
    "topic_4": "représentation",
    "topic_5": "santé",
    "topic_6": "symbolique-europe",
    "topic_7": "budget",
    "topic_8": "intégration",
    "topic_9": "PE",
    "topic_10": "droits (femmes-minorités)",
    "topic_11": "vie parlementaire",
    "topic_12": "mémoire",
    "topic_13": "antisémitisme",
    
}

df_binary = df_binary.rename(columns=theme_names)

## Tenter version solide en local

In [ ]:
import math
import numpy as np
import os
import traceback
from datasets import Dataset, load_from_disk
from sentence_transformers import SentenceTransformer
import torch
from torch.cuda import is_available as cuda_available
from torch.cuda import synchronize, empty_cache
from gc import collect as gc_collect
import pandas as pd

# ---------- Config ----------
DATASET_PATH = "path/to/your/dataset"   # dossier créé par save_to_disk
OUT_DIR = "embeddings_output"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # adapte
BATCH_SIZE = 64
CHUNK_SIZE_TOKENS = 256   # taille de fenêtre (tokens) pour chunking
OVERLAP_TOKENS = 32
USE_NORMALIZE = True
DEVICE = "cuda" if cuda_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Helpers ----------
def simple_preprocess(texts):
    """Nettoyage basique: supprime None, strip, lowercase"""
    out = []
    for t in texts:
        if t is None: 
            out.append("")   # ou skip
        else:
            out.append(str(t).strip())
    return out

def chunk_text(text, tokenizer, chunk_size=256, overlap=32):
    """Découpe `text` en morceaux de tokens compatibles avec tokenizer. 
       Retourne list[str] de chunks (reconstruit en substrings via tokenizer.decode)."""
    # Tokenize into ids
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        piece = tokenizer.decode(tokens[start:end], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        chunks.append(piece)
        if end == len(tokens):
            break
        start = end - overlap
    return chunks

def texts_to_chunks(texts, tokenizer, chunk_size, overlap):
    """Pour une liste de textes, construit une liste de chunks et mémorise les indexs
       pour ré-agréger."""
    all_chunks = []
    mapping = []  # mapping[i] = (start_index_in_all_chunks, num_chunks_for_text_i)
    idx = 0
    for t in texts:
        chunks = chunk_text(t, tokenizer, chunk_size, overlap)
        all_chunks.extend(chunks)
        mapping.append((idx, len(chunks)))
        idx += len(chunks)
    return all_chunks, mapping

def aggregate_embeddings(mapping, all_embeddings, method="mean"):
    """Recrée une embedding par texte à partir des embeddings de chunks.
       mapping: list of (start, count). all_embeddings: np.array (N_chunks, dim)."""
    outs = []
    for start, count in mapping:
        if count == 0:
            outs.append(np.zeros(all_embeddings.shape[1], dtype=np.float32))
        else:
            chunk_embs = all_embeddings[start:start+count]
            if method == "mean":
                emb = chunk_embs.mean(axis=0)
            elif method == "max":
                emb = chunk_embs.max(axis=0)
            else:
                emb = chunk_embs.mean(axis=0)
            outs.append(emb)
    return np.vstack(outs)

# ---------- Load dataset ----------
ds = load_from_disk(DATASET_PATH)  # ou Dataset.load_from_disk
texts_raw = ds["texts"]   # adapte le nom de la colonne
texts = simple_preprocess(texts_raw)

# ---------- Init modèle ----------
model = SentenceTransformer(MODEL_NAME, device=DEVICE, trust_remote_code=False)
tokenizer = model.tokenizer  # tokenizer huggingface du modèle
# Optionnel : fixer max_seq_length si tu veux tronquer
model.max_seq_length = min(model.max_seq_length, 512)  # adapte

# ---------- Chunking si nécessaire ----------
# Si tu as beaucoup de textes courts, tu peux ignorer le chunking et encoder directement
need_chunking = any(len(tokenizer.encode(t, add_special_tokens=False)) > model.max_seq_length for t in texts)
if need_chunking:
    print("Chunking texts because some exceed max_seq_length...")
    all_chunks, mapping = texts_to_chunks(texts, tokenizer, CHUNK_SIZE_TOKENS, OVERLAP_TOKENS)
    to_encode = all_chunks
else:
    to_encode = texts
    mapping = [(i,1) for i in range(len(texts))]

# ---------- Encode par batch ----------
def batched_encode(model, items, batch_size=BATCH_SIZE, normalize=USE_NORMALIZE, device=DEVICE):
    embeddings = []
    for i in range(0, len(items), batch_size):
        batch = items[i:i+batch_size]
        emb = model.encode(batch, batch_size=len(batch), device=device, normalize_embeddings=normalize, convert_to_numpy=True, show_progress_bar=False)
        embeddings.append(emb)
    return np.vstack(embeddings)

try:
    all_embeddings = batched_encode(model, to_encode, batch_size=BATCH_SIZE, normalize=USE_NORMALIZE, device=DEVICE)
    # Si chunking, agréger
    if need_chunking:
        final_embeddings = aggregate_embeddings(mapping, all_embeddings, method="mean")
    else:
        final_embeddings = all_embeddings  # shape (N_texts, dim)

    # Sauvegarde efficace: numpy memmap + dataset parquet
    emb_path = os.path.join(OUT_DIR, "embeddings.npy")
    np.save(emb_path, final_embeddings)  # petite dataset ok
    # Si très grand: np.memmap + écriture progressive serait plus adapté

    # Ajout au dataset (convert to list)
    ds2 = ds.add_column("embedding", [row.tolist() for row in final_embeddings])
    ds2.save_to_disk(os.path.join(OUT_DIR, "dataset_with_embeddings"))
    print("Saved embeddings and dataset successfully.")
except Exception as e:
    print("Error during encoding:")
    traceback.print_exc()
finally:
    # cleanup
    del model
    if DEVICE.startswith("cuda"):
        empty_cache()
        try:
            synchronize()
        except Exception:
            pass
    gc_collect()
